# Notebook 2: Developing estat.municipal_code_download_csv()

Created: 2026-04-28

### The Task

Develop function(s) to download and save the municipal codes as was done in Notebook 1. Use the list of population census years, but allow users to substitute their own list of years. Also enable grabbing the pandas.DataFrame without saving it as a CSV file.

Run the following in the terminal to install this package with its current functions and constants.

```
pip install .
```

In [1]:
import os
def get_relative_data_path():
    datapath = os.path.join(os.path.split(os.environ['VIRTUAL_ENV'])[0], 'data')
    reltodata = os.path.relpath(datapath,os.path.curdir)
    return reltodata

get_relative_data_path()

'..\\data'

### Set up for coding

Do the following in a terminal and in the Python .venv environment.

```
pip install python-dotenv
dotenv set ESTAT_APP_ID your-app-id
pip install estatjp
```

## Update full set of municipal codes with Japanese and English names

Instead of 'manually' selecting for census years as was done in Notebook 1, the routine below gets the municipal codes for all census years. These can then be used for datasets from other years as well.

### Get list of survey years

Do this by downloading a table containing the population of Tokyo's 23-ku area by year. The choice of area is somewhat arbitrary, but does need to be consistent for the whole time range. The 100,000 cell limit for API downloads prevents getting the population for all 3818 areas covered.

In [2]:
# For the available years from https://www.e-stat.go.jp/en/dbview?sid=0000020101
# http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdArea=13100&cdCat01=A1101&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0
yearsapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdArea=13100&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

import datetime
from estatjp import api
from estatjp import exceptions as xs
import pandas as pd
try:
    dfs_years = api.get_csv_data(yearsapiurlen,description=datetime.datetime.now())
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

print(dfs_years.get('Main'))

  tab_code  Observation Value cat01_code          A Population and Households  \
0    00001  Observation value      A1101  A1101_Total population (Both sexes)   
1    00001  Observation value      A1101  A1101_Total population (Both sexes)   
2    00001  Observation value      A1101  A1101_Total population (Both sexes)   
3    00001  Observation value      A1101  A1101_Total population (Both sexes)   
4    00001  Observation value      A1101  A1101_Total population (Both sexes)   
5    00001  Observation value      A1101  A1101_Total population (Both sexes)   
6    00001  Observation value      A1101  A1101_Total population (Both sexes)   
7    00001  Observation value      A1101  A1101_Total population (Both sexes)   
8    00001  Observation value      A1101  A1101_Total population (Both sexes)   

  area_code              AREA   time_code SURVEY YEAR    unit    value  \
0     13100  Tokyo-to Ku-area  1980100000        1980  person  8351893   
1     13100  Tokyo-to Ku-area  1985100000

### Get populations for all areas

Do this one census at a time. Again, the populations aren't necessary for our task but we have to retrieve something to get the names and codes of the municipalities. The loop below inserts the time_code after the "cdTime=" filter in the api url.

In [3]:
# For the municipal codes in English from https://www.e-stat.go.jp/en/dbview?sid=0000020101
codeapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdTime=&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

# For the municipal codes in Japanese from https://www.e-stat.go.jp/dbview?sid=0000020101
codeapiurlja = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdTime=&appId=&lang=J&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

# Create empty DataFrame
columns = ['census_year','muni_code','muni_name_en','muni_name_ja','population']
df_result = pd.DataFrame(columns=columns)
dftimecodes = dfs_years.get('Main')['time_code'].astype(str)
for tc in dftimecodes:
    url_split = codeapiurlen.split("cdTime=")
    if len(url_split) != 2:
        err_msg = "From CheckUrl: " + codeapiurlen
        try:
            raise xs.AppIDError()
        except xs.AppIDError as e:
            e.add_note(err_msg)
            raise
    urlloop = url_split[0] + "cdTime=" + tc + url_split[1]
    try:
        dfs_en = api.get_csv_data(urlloop,description=datetime.datetime.now())
    except Exception as e:
        print(type(e))
        print(e.args)
        print(e.with_traceback)

    url_split = codeapiurlja.split("cdTime=")
    if len(url_split) != 2:
        err_msg = "From CheckUrl: " + codeapiurlja
        try:
            raise xs.AppIDError()
        except xs.AppIDError as e:
            e.add_note(err_msg)
            raise
    urlloop = url_split[0] + "cdTime=" + tc + url_split[1]
    try:
        dfs_ja = api.get_csv_data(urlloop,description=datetime.datetime.now())
    except Exception as e:
        print(type(e))
        print(e.args)
        print(e.with_traceback)

    maindf_en = dfs_en.get('Main')
    maindf_en['area_code'] = maindf_en['area_code'].apply(lambda x: f"{x:05}")

    maindf_ja = dfs_ja.get('Main')
    maindf_ja['area_code'] = maindf_en['area_code'].apply(lambda x: f"{x:05}")

    maindf = pd.merge(maindf_en[["SURVEY YEAR","time_code","area_code","AREA","value"]],maindf_ja[["調査年","time_code","area_code","地域","value"]], on=["time_code","area_code","value"] )
    df_loop = maindf[["SURVEY YEAR","area_code","AREA","地域","value"]]
    df_loop.columns = ['census_year','muni_code','muni_name_en','muni_name_ja','population']

    df_result = pd.concat([df_result, df_loop], ignore_index = True)

# write to file
df_result.to_csv("../data/municipal-codes.csv", index=False)
df_result

,census_year,muni_code,muni_name_en,muni_name_ja,population
0,1980,01100,Hokkaido Sapporo-shi,北海道 札幌市,1401757
1,1980,01101,Hokkaido Sapporo-shi Chuo-ku,北海道 札幌市 中央区,181806
2,1980,01102,Hokkaido Sapporo-shi Kita-ku,北海道 札幌市 北区,195370
3,1980,01103,Hokkaido Sapporo-shi Higashi-ku,北海道 札幌市 東区,213310
4,1980,01104,Hokkaido Sapporo-shi Shiroishi-ku,北海道 札幌市 白石区,228061
...,...,...,...,...,...
25058,2020,47361,Okinawa-ken Kumejima-cho,沖縄県 久米島町,7192
25059,2020,47362,Okinawa-ken Yaese-cho,沖縄県 八重瀬町,30941
25060,2020,47375,Okinawa-ken Tarama-son,沖縄県 多良間村,1058
25061,2020,47381,Okinawa-ken Taketomi-cho,沖縄県 竹富町,3942
